<a href="https://colab.research.google.com/github/adeniasfilho/Assistente_De_Voz_Multi_Idiomas_Whisper-ChatGPT/blob/main/Assistente_De_Voz_Multi_Idiomas_Whisper_ChatGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
language = 'pt'

In [23]:
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})

var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({audio:true})
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""
def record(sec=5):
  display(Javascript(RECORD))
  js_result = output.eval_js('record(%s)'% (sec * 1000))
  audio = b64decode(js_result.split(',')[1])
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
    return f'/content/{file_name}' # Corrected: Removed the space before {file_name}
print("Ouvindo...\n")
record_file = record()
display(Audio(record_file, autoplay=True))



Ouvindo...



<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/request_audio.wav')

Reconhecimento de Voz com Whisper


In [ ]:
#!pip install --upgrade openai
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.4 MB/s eta 0:00:00


In [24]:
import whisper
model = whisper.load_model("small")
result = model.transcribe(record_file, fp16=False,language=language)
transcription=result["text"]
print(transcription)

 Calendário Nacional de Vacinação 2026


3. Integração com a API do ChatGpt


In [ ]:
!pip install openai

In [ ]:
import openai
import os
from openai import OpenAI

os.environ['OPENAI_API_KEY'] = 'TODO'
openai.api_key = os.environ.get('OPENAI_API_KEY')

#Versão Legada
'''
client = openai.OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
response = openai.ChatCompletion.create(
    model="gpt-4",
    messages= [{"role":"user", "content":transcription}]
)
'''
#Versão Atual
client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": transcription}]
)

chatgpt_response = response.choices[0].message.content
print(chatgpt_response)

4.Sintentizando a Resposta do ChatGPT Com Voz(gTTS)


In [ ]:
!pip install gTTS
from gtts import gTTS

gtts_object=gTTS(text=chatgpt_response,lang=language,slow=False)
response_audio="/content/response_audio.wav"
gtts_object.save(response_audio)
display(Audio(response_audio,autoplay=True))